In [1]:
from eeg_music.data import EEGMusicDataset
from pathlib import Path
# ds = EEGMusicDataset.load_ondisk(Path("./datasets/bcmi_preprocessed/bcmi_full_ica_40ch/"))
# onesubj_ds = EEGMusicDataset.load_ondisk(Path("./datasets/bcmi_preprocessed/bcmi_onesubj_ica_40ch/"))
ds = EEGMusicDataset.load_ondisk(Path("./datasets/musin_full_ica_40ch/"))
len(ds)

/home/zmrocze/studia/uwr/eeg-magisterka/.devenv/state/venv/lib/python3.12/site-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()


240

In [ ]:
splitted = ds.subject_wise_split(p_train=0.7, p_val=0.0)
train_ds, test_ds = splitted["train"], splitted["test"]
len(train_ds), len(test_ds)

In [ ]:
from fractions import Fraction
from eeg_music.data import temporal_train_test_split
from eeg_music.data import ArrayStratifiedSamplingDataset, temporal_train_test_split
from eeg_music.data import RepeatedDataset
import numpy as np

train_ds, test_ds = temporal_train_test_split(ds, length_sec=Fraction(20, 1))

In [ ]:
from fractions import Fraction

def create_X_y(dataset):
    X = []
    y = []
    for i in range(len(dataset)):
        sample = dataset[i]
        X.append(sample.eeg_data.get_array().data)

        y.append(sample.music_data.music_id.song_id - 1)
        
    return np.array(X), np.array(y)

def create_wrapped_ds(ds, repeat=1, trial_length_secs=Fraction(3, 1)):
  return RepeatedDataset(ArrayStratifiedSamplingDataset(ds, 10, trial_length_secs=trial_length_secs), repeat)

def create_split(dataset):
  train_ds_repeated = create_wrapped_ds(train_ds, repeat=10)
  test_ds_repeated = create_wrapped_ds(test_ds, repeat=10)
  X_train, y_train = create_X_y(train_ds_repeated)
  X_test, y_test = create_X_y(test_ds_repeated)

  return (X_train, y_train), (X_test, y_test)


In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression


def experiment(split):
  (X_train, y_train), (X_test, y_test) = split
  # Flatten EEG data for traditional ML models (samples, channels, timepoints) -> (samples, features)
  X_train_flat = X_train.reshape(X_train.shape[0], -1)
  X_test_flat = X_test.reshape(X_test.shape[0], -1)

  print(f"Training set: {X_train_flat.shape}, Labels: {y_train.shape}")
  print(f"Test set: {X_test_flat.shape}, Labels: {y_test.shape}")
  print(f"Number of classes: {len(np.unique(y_train))}")

  for model, model_name in [
    (XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1), "XGBoost"),
    (SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42), "SVM"),
    (KNeighborsClassifier(n_neighbors=5, n_jobs=-1), "KNN"),
    (LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1), "Linear")
    ]:

    # Train XGBoost
    print(f"Training {model_name}...")
    model.fit(X_train_flat, y_train)

    # Predict and evaluate
    y_pred_xgb = model.predict(X_test_flat)
    xgb_accuracy = accuracy_score(y_test, y_pred_xgb)

    print(f"{model_name} Test Accuracy: {xgb_accuracy:.4f}")
    print(f"{model_name} Classification Report:")
    print(classification_report(y_test, y_pred_xgb))
